In [1]:
import os
import pandas as pd
import polars as pl
import numpy as np
from prophet import Prophet
from datetime import timedelta
import kaggle_evaluation.mitsui_inference_server
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
train_labels_df = pd.read_csv('/kaggle/input/mitsui-commodity-prediction-challenge/train_labels.csv')
lagged_test_lab_4_df = pd.read_csv('/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_4.csv')

In [3]:
train_labels_df.tail(20)

,date_id,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,...,target_414,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423
1941,1941,0.000078,-0.003507,0.020287,0.014108,-0.009511,0.000911,-0.003814,-0.023056,-0.008897,...,NaN,-0.011525,NaN,0.022113,0.015484,0.002498,NaN,NaN,0.020213,NaN
1942,1942,0.004502,0.005992,-0.007980,-0.007661,0.007018,0.002437,0.002254,0.012562,0.002598,...,0.011833,-0.014624,-0.009990,0.024125,0.015996,0.005752,0.006341,0.010535,0.015526,-0.058521
1943,1943,0.005253,-0.003070,-0.003318,-0.003200,-0.013299,-0.040109,0.007924,-0.008450,-0.002596,...,0.024074,-0.014354,-0.005876,0.029499,0.014689,0.009475,0.011455,-0.005665,0.027233,-0.026371
1944,1944,NaN,NaN,0.000599,0.003907,-0.003859,0.004915,0.003107,NaN,NaN,...,-0.000145,-0.009604,-0.027801,-0.034384,0.004697,-0.000428,0.001967,-0.020739,-0.024410,-0.006881
1945,1945,NaN,NaN,0.011053,0.008216,-0.002612,0.002031,-0.004354,NaN,NaN,...,NaN,0.003697,NaN,-0.012088,-0.013203,-0.005897,NaN,NaN,-0.019112,NaN
1946,1946,0.002097,0.007676,-0.017297,-0.008476,-0.009235,-0.009344,0.004927,0.013470,-0.008573,...,0.020801,0.007732,-0.023085,0.012203,-0.012438,-0.013108,0.019849,0.000161,0.013699,0.016334
1947,1947,0.005957,-0.006200,-0.024532,-0.004002,0.012056,0.029059,-0.004295,0.006274,0.007338,...,0.015494,0.022658,-0.030251,-0.000618,0.003688,-0.006769,0.014615,0.000281,-0.001010,-0.061193
1948,1948,0.001849,-0.007703,-0.005259,-0.008274,-0.001940,0.012833,-0.010081,0.006885,-0.003008,...,0.035039,0.022234,-0.002396,0.020389,0.016058,-0.009305,0.039110,0.010308,0.016867,-0.056145
1949,1949,-0.005017,-0.006052,0.009829,0.012234,-0.008801,-0.023372,-0.009342,-0.007974,-0.009900,...,0.025953,0.017638,-0.013480,0.013482,0.019225,-0.005730,0.029508,0.021797,0.022270,-0.079864
1950,1950,0.001778,-0.010972,-0.001196,-0.001126,-0.014067,-0.012107,-0.004959,-0.004146,-0.008003,...,0.012926,0.013299,0.005647,-0.000061,-0.009896,-0.005708,0.001499,0.029689,0.003182,-0.068592


In [4]:
lagged_test_lab_4_df.tail(20)

,date_id,target_318,target_319,target_320,target_321,target_322,target_323,target_324,target_325,target_326,...,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423,label_date_id
114,1946,0.000238,-0.003129,0.026804,0.007153,-0.009471,0.007036,-0.024693,NaN,NaN,...,-0.011525,NaN,0.022113,0.015484,0.002498,NaN,NaN,0.020213,NaN,1941
115,1947,0.001972,-0.011639,0.014014,0.001262,-0.008931,0.009839,-0.029463,-0.041240,-0.024819,...,-0.014624,-0.009990,0.024125,0.015996,0.005752,0.006341,0.010535,0.015526,-0.058521,1942
116,1948,0.002845,-0.015907,0.029452,0.000447,-0.011604,0.021014,-0.042059,-0.033730,-0.033146,...,-0.014354,-0.005876,0.029499,0.014689,0.009475,0.011455,-0.005665,0.027233,-0.026371,1943
117,1949,-0.000272,-0.002996,0.003296,-0.000355,0.000615,0.029206,0.026307,-0.055779,-0.017963,...,-0.009604,-0.027801,-0.034384,0.004697,-0.000428,0.001967,-0.020739,-0.024410,-0.006881,1944
118,1950,0.007469,-0.000735,-0.010804,-0.012536,0.013803,0.009696,0.022044,NaN,NaN,...,0.003697,NaN,-0.012088,-0.013203,-0.005897,NaN,NaN,-0.019112,NaN,1945
119,1951,0.002181,0.009217,-0.000598,-0.008518,0.018791,0.009950,0.000658,-0.087741,-0.025655,...,0.007732,-0.023085,0.012203,-0.012438,-0.013108,0.019849,0.000161,0.013699,0.016334,1946
120,1952,0.000521,0.001797,0.011584,-0.001168,0.028677,-0.008687,0.005245,-0.049585,-0.025784,...,0.022658,-0.030251,-0.000618,0.003688,-0.006769,0.014615,0.000281,-0.001010,-0.061193,1947
121,1953,-0.004877,-0.001305,0.042231,0.010549,0.023436,-0.031174,-0.018215,-0.018680,-0.047710,...,0.022234,-0.002396,0.020389,0.016058,-0.009305,0.039110,0.010308,0.016867,-0.056145,1948
122,1954,-0.011648,0.000076,0.043761,0.012482,0.022009,-0.026806,-0.021805,0.019791,-0.032400,...,0.017638,-0.013480,0.013482,0.019225,-0.005730,0.029508,0.021797,0.022270,-0.079864,1949
123,1955,-0.008895,-0.000755,0.011564,-0.009103,0.014634,-0.024905,-0.005864,-0.089193,-0.010601,...,0.013299,0.005647,-0.000061,-0.009896,-0.005708,0.001499,0.029689,0.003182,-0.068592,1950


In [5]:
lagged_test_lab_4_df.head(10)

,date_id,target_318,target_319,target_320,target_321,target_322,target_323,target_324,target_325,target_326,...,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423,label_date_id
0,1832,0.001082,-0.020023,0.064526,0.028037,0.004506,-0.023554,-0.029856,NaN,NaN,...,0.019701,NaN,-0.027030,0.043602,0.027982,NaN,NaN,0.002177,NaN,1827
1,1833,-0.001296,-0.015281,0.041474,0.009498,-0.005352,0.009397,-0.027311,0.046440,-0.009329,...,0.012081,-0.020068,0.002858,0.019154,0.019018,0.003875,-0.035202,0.011246,0.099241,1828
2,1834,-0.002375,-0.021250,0.041091,0.020435,-0.007740,0.010090,-0.018724,0.033513,0.018609,...,0.016166,-0.028919,-0.007297,0.033262,0.023174,-0.028512,-0.017900,-0.002096,0.121451,1829
3,1835,-0.001846,0.000320,0.016153,0.021035,-0.007409,0.023807,-0.008369,0.045870,0.010528,...,-0.007742,-0.018436,0.004691,0.013311,0.000589,-0.014500,-0.046444,0.009058,0.109246,1830
4,1836,-0.000929,-0.004576,0.004014,0.006592,-0.020097,0.030620,-0.009754,0.050258,0.017386,...,-0.018850,-0.025373,0.031197,0.005873,-0.005650,-0.022926,-0.027990,0.011267,0.091318,1831
5,1837,0.005104,-0.008375,0.041446,0.031493,-0.003528,0.005438,-0.030237,0.053343,0.001698,...,-0.005291,-0.026959,0.025416,0.036183,0.000457,0.012325,0.023296,0.022195,-0.045001,1832
6,1838,0.002868,0.016572,0.005648,0.010829,0.014545,-0.007490,0.003335,0.070443,-0.023820,...,-0.011752,-0.006096,0.024564,-0.014669,-0.026384,0.049811,-0.022768,0.014820,0.036332,1833
7,1839,0.008355,-0.001507,0.009344,-0.001917,0.002014,-0.004223,-0.002697,0.049877,-0.015616,...,-0.007847,-0.023073,0.019207,-0.007262,-0.003535,0.045942,0.006377,0.007368,0.004076,1834
8,1840,0.008006,-0.012012,0.028759,0.000839,-0.021888,0.005742,-0.005277,0.087626,-0.023915,...,-0.008596,-0.011148,0.013248,0.006783,0.012370,0.059634,0.012869,0.001956,0.011147,1835
9,1841,0.007025,0.002567,-0.017011,-0.017748,-0.010468,-0.004882,0.026783,0.083687,0.005502,...,-0.013517,0.006281,-0.003193,-0.020847,-0.000352,0.030961,0.006029,-0.020107,0.040021,1836


In [6]:
1965-1832

133

In [7]:
365*5.383561643835616

1965.0

In [8]:
round(365*0.383561643835616)

140

In [9]:
import os
import pandas as pd
import polars as pl
import numpy as np
from prophet import Prophet
from datetime import timedelta
import kaggle_evaluation.mitsui_inference_server
from warnings import filterwarnings
filterwarnings("ignore")


# ========================
# CONSTANTS
# ========================
NUM_TARGET_COLUMNS = 424
PROPHET_WEIGHT = 0.05
PROPHET_TRAINING_DAYS = 365

# ========================
# LOAD STATIC TRAIN DATA
# ========================
train_labels = pd.read_csv("/kaggle/input/mitsui-commodity-prediction-challenge/train_labels.csv")
test_labels_lag_1_df = pd.read_csv('/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_1.csv')
sel_cols = [col for col in train_labels.columns if col.startswith("target_")]
print(len(sel_cols))
train_labels["date_id"] = train_labels["date_id"].astype(np.uint16)
base_date = pd.Timestamp("2020-01-01")
train_labels["ds"] = base_date + pd.to_timedelta(train_labels["date_id"], unit="D")

# Compute missing values per target
missing_pct = train_labels[sel_cols].isnull().mean()

# ========================
# PRE-TRAIN PROPHET MODELS
# ========================
prophet_models = {}
targets_to_model = train_labels[sel_cols].columns.values

for target in targets_to_model:
    df_prophet = train_labels[["ds", target]].ffill().bfill().rename(columns={target: "y"})
    # df_prophet = df_prophet.head(len(train_labels)-146)
    # model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    model.fit(df_prophet)
    prophet_models[target] = model

print(prophet_models)



424


13:21:12 - cmdstanpy - INFO - Chain [1] start processing
13:21:12 - cmdstanpy - INFO - Chain [1] done processing
13:21:12 - cmdstanpy - INFO - Chain [1] start processing
13:21:12 - cmdstanpy - INFO - Chain [1] done processing
13:21:12 - cmdstanpy - INFO - Chain [1] start processing
13:21:12 - cmdstanpy - INFO - Chain [1] done processing
13:21:12 - cmdstanpy - INFO - Chain [1] start processing
13:21:12 - cmdstanpy - INFO - Chain [1] done processing
13:21:12 - cmdstanpy - INFO - Chain [1] start processing
13:21:13 - cmdstanpy - INFO - Chain [1] done processing
13:21:13 - cmdstanpy - INFO - Chain [1] start processing
13:21:13 - cmdstanpy - INFO - Chain [1] done processing
13:21:13 - cmdstanpy - INFO - Chain [1] start processing
13:21:13 - cmdstanpy - INFO - Chain [1] done processing
13:21:13 - cmdstanpy - INFO - Chain [1] start processing
13:21:13 - cmdstanpy - INFO - Chain [1] done processing
13:21:13 - cmdstanpy - INFO - Chain [1] start processing
13:21:13 - cmdstanpy - INFO - Chain [1]

{'target_0': <prophet.forecaster.Prophet object at 0x7cc564ebb290>, 'target_1': <prophet.forecaster.Prophet object at 0x7cc564ef11d0>, 'target_2': <prophet.forecaster.Prophet object at 0x7cc55fda8390>, 'target_3': <prophet.forecaster.Prophet object at 0x7cc564f4af90>, 'target_4': <prophet.forecaster.Prophet object at 0x7cc564f35790>, 'target_5': <prophet.forecaster.Prophet object at 0x7cc55fab25d0>, 'target_6': <prophet.forecaster.Prophet object at 0x7cc55fa8a750>, 'target_7': <prophet.forecaster.Prophet object at 0x7cc55fab23d0>, 'target_8': <prophet.forecaster.Prophet object at 0x7cc55f70c290>, 'target_9': <prophet.forecaster.Prophet object at 0x7cc55f9a2990>, 'target_10': <prophet.forecaster.Prophet object at 0x7cc55f70df90>, 'target_11': <prophet.forecaster.Prophet object at 0x7cc55f8e6250>, 'target_12': <prophet.forecaster.Prophet object at 0x7cc55f8e6d10>, 'target_13': <prophet.forecaster.Prophet object at 0x7cc55f92ee90>, 'target_14': <prophet.forecaster.Prophet object at 0x7cc5

In [10]:
# ----------------------------------------
# 予測モデルの推論用関数
# ----------------------------------------
def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pl.DataFrame | pd.DataFrame:
    
    # この関数内では、予測に必要な情報だけを利用する
    # GrandTruthを使わない。train_labels.csvを直接参照しない
    
    X = test.to_pandas()
    date_id = X["date_id"].iloc[0]
    
    # 予測対象の日付を計算
    current_date = base_date + pd.to_timedelta(date_id, unit="D")
    
    preds = {}
    
    # 既存のProphetモデルを使用して予測値を生成
    for col in sel_cols:
        if col in prophet_models:
            # 予測したい日付のデータフレームを作成
            future_df = pd.DataFrame({"ds": [current_date]})
            
            # 事前学習したProphetモデルで予測を行う
            yhat = prophet_models[col].predict(future_df)["yhat"].iloc[0]
            preds[col] = yhat
        else:
            # モデルが存在しない場合はゼロを返す
            preds[col] = 0.0
            
    # DataFrameに変換して返す
    return pl.DataFrame(preds).select(pl.all().cast(pl.Float64))

# ========================
# RUN SERVER
# ========================
inference_server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("/kaggle/input/mitsui-commodity-prediction-challenge/",))
 
